# Kalman Filter Validation Against OpenRocket Sim Data

Tests the C++ `KalmanFilter` class (via pybind11 bindings) against exported
OpenRocket simulation data. Synthetic sensor noise is injected into the truth
data to simulate realistic IMU / barometer readings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob, os

from kalman_filter import KalmanFilter

## Load OpenRocket Data

In [ ]:
sim_files = sorted(glob.glob(os.path.join("simdata", "*.csv")))
print("Found:", sim_files)

def load_sim(path):
    df = pd.read_csv(path, comment="#")
    df.columns = ["time", "alt", "vvel", "vacc"]
    return df

datasets = {os.path.basename(f).replace(".csv", ""): load_sim(f) for f in sim_files}
for name, df in datasets.items():
    print(f"{name}: {len(df)} rows, {df.time.iloc[-1]:.1f} s")

## Sensor Simulation

Generate synthetic IMU and barometer readings from the truth data:
- **IMU** runs at every sim timestep with additive Gaussian noise + a small bias.
- **Barometer** runs at a lower rate (e.g. 25 Hz) with its own noise.

In [ ]:
# tuneable sensor / filter parameters
BARO_RATE_HZ     = 25       # barometer sample rate
BARO_NOISE_STD   = 1.0      # baro noise std-dev (m)
IMU_NOISE_STD    = 0.5      # IMU accel noise std-dev (m/s^2)
IMU_BIAS_TRUE    = 0.3      # true accelerometer bias (m/s^2)

# filter tuning
STDDEV_A         = 1.0      # process noise: unmodelled accel std-dev
STDDEV_B         = 0.01     # process noise: bias drift rate std-dev
R_BARO           = BARO_NOISE_STD ** 2  # measurement noise variance

## Run Filter & Collect Results

In [ ]:
rng = np.random.default_rng(42)

results = {}  # name -> dict of arrays

for name, df in datasets.items():
    t    = df.time.values
    alt  = df.alt.values
    vvel = df.vvel.values
    vacc = df.vacc.values

    n = len(t)
    baro_period = 1.0 / BARO_RATE_HZ

    # synthetic sensor signals
    imu_readings  = vacc + IMU_BIAS_TRUE + rng.normal(0, IMU_NOISE_STD, n)
    baro_readings = alt + rng.normal(0, BARO_NOISE_STD, n)

    # init filter
    x_init = [0.0, 0.0, 0.0]
    P_init = [[10.0, 0, 0],
              [0, 10.0, 0],
              [0, 0, 1.0]]
    kf = KalmanFilter(x_init, P_init, R_BARO, STDDEV_A, STDDEV_B)

    est_h = np.zeros(n)
    est_v = np.zeros(n)
    est_b = np.zeros(n)
    baro_used = np.full(n, np.nan)  # NaN when baro not sampled

    last_baro_t = -baro_period  # force first baro update at t=0

    for i in range(n):
        dt = t[i] - t[i - 1] if i > 0 else 0.0
        if dt > 0:
            kf.predict(dt, imu_readings[i])

        # barometer update at ~BARO_RATE_HZ
        if t[i] - last_baro_t >= baro_period:
            kf.update(baro_readings[i])
            baro_used[i] = baro_readings[i]
            last_baro_t = t[i]

        est_h[i] = kf.h
        est_v[i] = kf.v
        est_b[i] = kf.b_a

    results[name] = dict(t=t, alt=alt, vvel=vvel, vacc=vacc,
                         est_h=est_h, est_v=est_v, est_b=est_b,
                         baro_used=baro_used)
    print(f"{name}  |  peak alt: truth={alt.max():.1f} m, est={est_h.max():.1f} m")

## Plots

In [ ]:
for name, r in results.items():
    fig, (ax_alt, ax_vel) = plt.subplots(1, 2, figsize=(20, 5), sharey=False)
    fig.suptitle(f"{name}", fontsize=14)

    # --- altitude ---
    ax_alt.plot(r["t"], r["alt"], label="Truth (OpenRocket)", linewidth=1.5)
    ax_alt.plot(r["t"], r["est_h"], label="Kalman estimate", linewidth=1.2, alpha=0.9)
    baro_mask = ~np.isnan(r["baro_used"])
    ax_alt.scatter(r["t"][baro_mask], r["baro_used"][baro_mask],
                   s=6, c="red", alpha=0.4, label="Baro readings", zorder=1)
    ax_alt.set_ylabel("Altitude (m)")
    ax_alt.set_xlabel("Time (s)")
    ax_alt.legend(loc="upper right")
    ax_alt.grid(True, alpha=0.3)

    ax2 = ax_alt.twinx()
    ax2.plot(r["t"], r["est_h"] - r["alt"], color="gray", linewidth=0.7, alpha=0.4)
    ax2.set_ylabel("Error (m)", color="gray")
    ax2.tick_params(axis="y", labelcolor="gray")

    # --- velocity ---
    ax_vel.plot(r["t"], r["vvel"], label="Truth", linewidth=1.5)
    ax_vel.plot(r["t"], r["est_v"], label="Kalman estimate", linewidth=1.2, alpha=0.9)
    ax_vel.set_ylabel("Vertical velocity (m/s)")
    ax_vel.set_xlabel("Time (s)")
    ax_vel.legend(loc="upper right")
    ax_vel.grid(True, alpha=0.3)

    ax2 = ax_vel.twinx()
    ax2.plot(r["t"], r["est_v"] - r["vvel"], color="gray", linewidth=0.7, alpha=0.4)
    ax2.set_ylabel("Error (m/s)", color="gray")
    ax2.tick_params(axis="y", labelcolor="gray")

    plt.tight_layout()
    plt.show()

## Monte Carlo Parameter Sweep

Randomly samples the three filter tuning parameters (`stddev_a`, `stddev_b`, `R_baro`) over a log-uniform grid and evaluates combined altitude + velocity RMSE across all simulation datasets. The sample with the lowest combined RMSE is taken as the best.

In [ ]:
def run_filter(df, stddev_a, stddev_b, r_baro, seed=42):
    """Run the C++ KalmanFilter on one dataset; return (alt_rmse, vel_rmse)."""
    rng_s = np.random.default_rng(seed)
    t, alt, vvel, vacc = df.time.values, df.alt.values, df.vvel.values, df.vacc.values
    n = len(t)
    baro_period = 1.0 / BARO_RATE_HZ

    imu_readings  = vacc + IMU_BIAS_TRUE + rng_s.normal(0, IMU_NOISE_STD, n)
    baro_readings = alt  + rng_s.normal(0, BARO_NOISE_STD, n)

    kf = KalmanFilter([0.0, 0.0, 0.0],
                      [[10.0, 0, 0], [0, 10.0, 0], [0, 0, 1.0]],
                      r_baro, stddev_a, stddev_b)
    est_h = np.zeros(n)
    est_v = np.zeros(n)
    last_baro_t = -baro_period

    for i in range(n):
        dt = t[i] - t[i - 1] if i > 0 else 0.0
        if dt > 0:
            kf.predict(dt, imu_readings[i])
        if t[i] - last_baro_t >= baro_period:
            kf.update(baro_readings[i])
            last_baro_t = t[i]
        est_h[i] = kf.h
        est_v[i] = kf.v

    return (np.sqrt(np.mean((est_h - alt) ** 2)),
            np.sqrt(np.mean((est_v - vvel) ** 2)))


N_SAMPLES = 1000
rng_mc = np.random.default_rng(0)

sa_samples = np.exp(rng_mc.uniform(np.log(0.05),  np.log(10.0),  N_SAMPLES))
sb_samples = np.exp(rng_mc.uniform(np.log(0.001), np.log(0.5),   N_SAMPLES))
rb_samples = np.exp(rng_mc.uniform(np.log(0.1),   np.log(25.0),  N_SAMPLES))

rows = []
for sa, sb, rb in zip(sa_samples, sb_samples, rb_samples):
    alt_sum = vel_sum = 0.0
    for df in datasets.values():
        ar, vr = run_filter(df, sa, sb, rb)
        alt_sum += ar
        vel_sum += vr
    n_ds = len(datasets)
    rows.append(dict(stddev_a=sa, stddev_b=sb, r_baro=rb,
                     alt_rmse=alt_sum / n_ds,
                     vel_rmse=vel_sum / n_ds,
                     combined=(alt_sum + vel_sum) / n_ds))

mc = pd.DataFrame(rows)
best = mc.loc[mc.combined.idxmin()]

print(f"Best parameters ({N_SAMPLES} samples):")
print(f"  stddev_a = {best.stddev_a:.4f} m/sÂ²")
print(f"  stddev_b = {best.stddev_b:.5f} m/sÂ³")
print(f"  r_baro   = {best.r_baro:.4f} mÂ²")
print(f"  alt RMSE = {best.alt_rmse:.4f} m")
print(f"  vel RMSE = {best.vel_rmse:.4f} m/s")
print(f"  combined = {best.combined:.4f}")

## Sweep Visualisations

In [ ]:
clim  = (mc.combined.min(), mc.combined.quantile(0.95))
cols  = ["stddev_a",       "stddev_b",       "r_baro"]
xlbls = ["stddev_a (m/s²)", "stddev_b (m/s³)", "R_baro (m²)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
fig.suptitle("Parameter Sensitivity", fontsize=13)

for ax, col, xlabel in zip(axes, cols, xlbls):
    sc = ax.scatter(mc[col], mc.combined, c=mc.combined,
                    cmap="viridis_r", s=12, alpha=0.5,
                    vmin=clim[0], vmax=clim[1])
    ax.axvline(best[col], color="red", linestyle="--", linewidth=1.5,
               label=f"best = {best[col]:.3f}")
    ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Combined RMSE (m)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.colorbar(sc, ax=axes, label="Combined RMSE (m)", shrink=0.8)
plt.show()

In [ ]:
pairs  = [("stddev_a", "stddev_b"), ("stddev_a", "r_baro"), ("stddev_b", "r_baro")]
xlbls  = {"stddev_a": "stddev_a (m/s²)", "stddev_b": "stddev_b (m/s³)", "r_baro": "R_baro (m²)"}
clim   = (mc.combined.min(), mc.combined.quantile(0.95))

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
fig.suptitle("Parameter Space (colored by Combined RMSE)", fontsize=13)

for ax, (cx, cy) in zip(axes, pairs):
    sc = ax.scatter(mc[cx], mc[cy], c=mc.combined, cmap="viridis_r",
                    s=12, alpha=0.5, vmin=clim[0], vmax=clim[1])
    ax.scatter(best[cx], best[cy], c="red", s=120, marker="*",
               zorder=5, label="best", edgecolors="white", linewidths=0.5)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(xlbls[cx])
    ax.set_ylabel(xlbls[cy])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.colorbar(sc, ax=axes, label="Combined RMSE (m)", shrink=0.8)
plt.show()

In [ ]:
for name, df in datasets.items():
    rng_s = np.random.default_rng(42)
    t, alt, vvel, vacc = df.time.values, df.alt.values, df.vvel.values, df.vacc.values
    n = len(t)
    baro_period = 1.0 / BARO_RATE_HZ

    imu_readings  = vacc + IMU_BIAS_TRUE + rng_s.normal(0, IMU_NOISE_STD, n)
    baro_readings = alt  + rng_s.normal(0, BARO_NOISE_STD, n)

    kf = KalmanFilter([0.0, 0.0, 0.0],
                      [[10.0, 0, 0], [0, 10.0, 0], [0, 0, 1.0]],
                      best.r_baro, best.stddev_a, best.stddev_b)
    est_h = np.zeros(n)
    est_v = np.zeros(n)
    baro_used = np.full(n, np.nan)
    last_baro_t = -baro_period

    for i in range(n):
        dt = t[i] - t[i - 1] if i > 0 else 0.0
        if dt > 0:
            kf.predict(dt, imu_readings[i])
        if t[i] - last_baro_t >= baro_period:
            kf.update(baro_readings[i])
            baro_used[i] = baro_readings[i]
            last_baro_t = t[i]
        est_h[i] = kf.h
        est_v[i] = kf.v

    alt_rmse = np.sqrt(np.mean((est_h - alt) ** 2))
    vel_rmse = np.sqrt(np.mean((est_v - vvel) ** 2))

    fig, (ax_alt, ax_vel) = plt.subplots(1, 2, figsize=(20, 5))
    fig.suptitle(f"{name} Best Parameters  "
                 f"(alt RMSE {alt_rmse:.3f} m, vel RMSE {vel_rmse:.3f} m/s)", fontsize=13)

    ax_alt.plot(t, alt, label="Truth (OpenRocket)", linewidth=1.5)
    ax_alt.plot(t, est_h, label="Kalman estimate", linewidth=1.2, alpha=0.9)
    bm = ~np.isnan(baro_used)
    ax_alt.scatter(t[bm], baro_used[bm], s=6, c="red", alpha=0.4,
                   label="Baro readings", zorder=1)
    ax_alt.set_ylabel("Altitude (m)")
    ax_alt.set_xlabel("Time (s)")
    ax_alt.legend(loc="upper right")
    ax_alt.grid(True, alpha=0.3)
    ax2 = ax_alt.twinx()
    ax2.plot(t, est_h - alt, color="gray", linewidth=0.7, alpha=0.4)
    ax2.set_ylabel("Error (m)", color="gray")
    ax2.tick_params(axis="y", labelcolor="gray")

    ax_vel.plot(t, vvel, label="Truth", linewidth=1.5)
    ax_vel.plot(t, est_v, label="Kalman estimate", linewidth=1.2, alpha=0.9)
    ax_vel.set_ylabel("Vertical velocity (m/s)")
    ax_vel.set_xlabel("Time (s)")
    ax_vel.legend(loc="upper right")
    ax_vel.grid(True, alpha=0.3)
    ax2 = ax_vel.twinx()
    ax2.plot(t, est_v - vvel, color="gray", linewidth=0.7, alpha=0.4)
    ax2.set_ylabel("Error (m/s)", color="gray")
    ax2.tick_params(axis="y", labelcolor="gray")

    plt.tight_layout()
    plt.show()